# Knowledge Graph: FRED Economic Indicators Analysis

## Overview
This notebook creates a comprehensive knowledge graph of FRED API economic indicators, analyzing:
1. **Correlation patterns** between most popular FRED indicators
2. **Temporal relationships** and lag analysis
3. **Network structure** showing which indicators drive others
4. **Integration with stock indices** and large-cap stocks
5. **Dynamic knowledge graph** visualization

**Author:** Generated Analysis  
**Date:** `r Sys.Date()`  
**Data Sources:** FRED API, Yahoo Finance, Alpha Vantage

In [ ]:
# Set working directory and options
setwd(".")
knitr::opts_chunk$set(echo = TRUE, message = FALSE, warning = FALSE, fig.width = 12, fig.height = 8)
options(scipen = 999)

In [ ]:
library(renv)
renv::init()

In [ ]:
# Install/load required packages for visualization
if (!require(networkD3)) install.packages("networkD3")
if (!require(visNetwork)) install.packages("visNetwork")
if (!require(ggraph)) install.packages("ggraph")
if (!require(igraph)) install.packages("igraph")

library(networkD3)
library(visNetwork)
library(ggraph)
library(igraph)
library(ggplot2)

In [ ]:
# Load required libraries
if (!require(pacman)) { install.packages("pacman") }
pacman::p_load(
    fredr,          # FRED API
    dplyr,          # Data manipulation
    tidyr,          # Data reshaping
    tibble,         # For column_to_rownames function
    lubridate,      # Date handling
    ggplot2,        # Plotting
    visNetwork,     # Network visualization
    igraph,         # Graph analysis
    corrplot,       # Correlation plots
    quantmod,       # Financial data
    data.table,     # Fast data operations
    DT,             # Interactive tables
    plotly,         # Interactive plots
    networkD3,      # D3 network plots
    scales,         # Scale functions
    RColorBrewer,   # Color palettes
    tidyquant,      # Financial analysis
    PerformanceAnalytics, # Financial performance
    psych           # Statistical functions
)

In [ ]:
# Set up FRED API key
if (file.exists("Renviron.site")) {
    readRenviron(path = "Renviron.site")
    fredr::fredr_set_key(key = Sys.getenv("FRED_API"))
} else {
    # Fallback - user should set their own key
    cat("Please set your FRED API key using: fredr::fredr_set_key('your_key_here')\n")
}

## 1. Most Popular FRED Economic Indicators

We'll analyze the most commonly used economic indicators across different categories:

In [ ]:
# Define comprehensive list of 100+ popular FRED indicators
fred_indicators <- list(
    # Employment & Labor (20 indicators)
    employment = c(
        "PAYEMS",         # All Employees: Total Nonfarm
        "UNRATE",         # Unemployment Rate
        "CIVPART",        # Labor Force Participation Rate
        "ICSA",           # Initial Claims
        "AWHAETP",        # Average Weekly Hours
        "CES0500000003",  # Average Hourly Earnings
        "NPPTTL",         # Nonfarm Private Payroll Employment
        "UEMPMED",        # Median Duration of Unemployment
        "U6RATE",         # Total unemployed, plus all marginally attached workers
        "JTSJOL",         # Job Openings: Total Nonfarm
        "JTSQUR",         # Quits: Total Nonfarm
        "LREM64TTUSM156S", # Labor Force Participation Rate - Men
        "LNS11300002",    # Labor Force Participation Rate - Women
        "AWHNONAG",       # Average Weekly Hours of Production Employees: Total Private
        "CES0500000008",  # Average Weekly Earnings of Production Employees: Total Private
        "UNEMPLOY",       # Unemployed
        "EMRATIO",        # Employment-Population Ratio
        "LNS14000031",    # Unemployment Rate - College Graduates
        "LNS14000025",    # Unemployment Rate - Less than High School
        "LREM25TTUSM156S" # Labor Force Participation Rate 25-54 years
    ),

    # Inflation & Prices (18 indicators)
    inflation = c(
        "CPIAUCSL",       # Consumer Price Index for All Urban Consumers: All Items
        "CPILFESL",       # Consumer Price Index for All Urban Consumers: All Items Less Food and Energy
        "PCEPI",          # Personal Consumption Expenditures: Chain-type Price Index
        "PCEPILFE",       # Personal Consumption Expenditures Excluding Food and Energy
        "T10YIE",         # 10-Year Breakeven Inflation Rate
        "T5YIE",          # 5-Year Breakeven Inflation Rate
        "T5YIFR",         # 5-Year, 5-Year Forward Inflation Expectation Rate
        "CPIENGSL",       # Consumer Price Index for All Urban Consumers: Energy
        "CPIUFDSL",       # Consumer Price Index for All Urban Consumers: Food
        "CPIHOSNS",       # Consumer Price Index for All Urban Consumers: Housing
        "CUSR0000SEHC",   # Consumer Price Index for All Urban Consumers: Medical Care
        "CUSR0000SAT1",   # Consumer Price Index for All Urban Consumers: Apparel
        "CUSR0000SETB",   # Consumer Price Index for All Urban Consumers: Transportation
        "DPCCRG3M086SBEA", # Personal Consumption Expenditures: Goods
        "DPCCRV3M086SBEA", # Personal Consumption Expenditures: Services
        "PPIFIS",         # Producer Price Index by Commodity: Finished Goods
        "PPIACO",         # Producer Price Index: All Commodities
        "GOLDAMGBD228NLBM" # Gold Fixing Price 10:30 A.M. (London time) in London Bullion Market
    ),

    # Interest Rates & Monetary Policy (22 indicators)
    rates = c(
        "DFF",            # Effective Federal Funds Rate
        "DGS10",          # 10-Year Treasury Constant Maturity Rate
        "DGS2",           # 2-Year Treasury Constant Maturity Rate
        "DGS1",           # 1-Year Treasury Constant Maturity Rate
        "DGS30",          # 30-Year Treasury Constant Maturity Rate
        "DGS5",           # 5-Year Treasury Constant Maturity Rate
        "DGS3MO",         # 3-Month Treasury Constant Maturity Rate
        "DGS6MO",         # 6-Month Treasury Constant Maturity Rate
        "MORTGAGE30US",   # 30-Year Fixed Rate Mortgage Average in the United States
        "MORTGAGE15US",   # 15-Year Fixed Rate Mortgage Average in the United States
        "TB3MS",          # 3-Month Treasury Bill: Secondary Market Rate
        "GS1",            # 1-Year Treasury Constant Maturity Rate
        "GS2",            # 2-Year Treasury Constant Maturity Rate
        "GS3",            # 3-Year Treasury Constant Maturity Rate
        "GS5",            # 5-Year Treasury Constant Maturity Rate
        "GS7",            # 7-Year Treasury Constant Maturity Rate
        "GS10",           # 10-Year Treasury Constant Maturity Rate
        "GS20",           # 20-Year Treasury Constant Maturity Rate
        "GS30",           # 30-Year Treasury Constant Maturity Rate
        "T10Y2Y",         # 10-Year Treasury Constant Maturity Minus 2-Year Treasury
        "T10Y3M",         # 10-Year Treasury Constant Maturity Minus 3-Month Treasury
        "REAINTRATREARAT10Y" # 10-Year Real Interest Rate
    ),

    # Money Supply & Credit (15 indicators)
    money = c(
        "M1SL",           # M1 Money Stock
        "M2SL",           # M2 Money Stock
        "BOGMBASE",       # St. Louis Adjusted Monetary Base
        "WALCL",          # All Federal Reserve Banks - Total Assets
        "TOTRESNS",       # Total Reserve Balances Maintained
        "DPSACBW027SBOG", # Deposits, All Commercial Banks
        "LOANS",          # Loans and Leases in Bank Credit, All Commercial Banks
        "CONSUMER",       # Consumer Loans at All Commercial Banks
        "REALLN",         # Real Estate Loans at All Commercial Banks
        "BUSLOANS",       # Commercial and Industrial Loans, All Commercial Banks
        "TOTLL",          # Total Loans and Leases at Commercial Banks
        "MULT",           # St. Louis Adjusted Monetary Base
        "AMBSL",          # St. Louis Adjusted Monetary Base
        "REQRESNS",       # Required Reserves of Depository Institutions
        "EXCSRESNS"       # Excess Reserves of Depository Institutions
    ),

    # Economic Output & Growth (18 indicators)
    output = c(
        "GDP",            # Gross Domestic Product
        "GDPC1",          # Real Gross Domestic Product
        "INDPRO",         # Industrial Production Index
        "NAPMEI",         # ISM Manufacturing: Employment Index
        "RSAFS",          # Advance Retail Sales: Retail Trade
        "HOUST",          # New Privately-Owned Housing Units Started: Total Units
        "CAPUTLB50001SQ", # Capacity Utilization: Manufacturing
        "TCU",            # Capacity Utilization: Total Industry
        "NAPMPI",         # ISM Manufacturing: PMI Composite Index
        "NAPMNOI",        # ISM Manufacturing: New Orders Index
        "NAPMII",         # ISM Manufacturing: Inventories Index
        "NAPMSI",         # ISM Manufacturing: Supplier Deliveries Index
        "RRSFS",          # Real Retail and Food Services Sales
        "CMRMTSPL",       # Real Manufacturing and Trade Industries Sales
        "DGORDER",        # Manufacturers' New Orders: Durable Goods
        "NEWORDER",       # Manufacturers' New Orders: Nondefense Capital Goods Excluding Aircraft
        "CPMNACBS",       # ISM Manufacturing: New Orders Index
        "IPMAN"           # Industrial Production: Manufacturing
    ),

    # Housing Market (12 indicators)
    housing = c(
        "CSUSHPINSA",     # S&P/Case-Shiller U.S. National Home Price Index
        "PERMIT",         # New Private Housing Units Authorized by Building Permits
        "HOUST1F",        # New Privately-Owned Housing Units Started: 1-Unit Structures
        "EXHOSLUSM495S",  # Existing Home Sales
        "ACTLISCOUUS",    # Active Listing Count in the United States
        "MSACSR",         # Monthly Supply of Houses in the United States
        "NHSDPTS",        # New Houses Sold in the United States
        "USSTHPI",        # All-Transactions House Price Index for the United States
        "COMPUTSA",       # Existing Home Sales
        "HSN1F",          # New One Family Houses Sold: United States
        "RHORUSQ156N",    # Homeownership Rate in the United States
        "TTLCONS"         # Total Construction Spending
    ),

    # Consumer & Business Sentiment (12 indicators)
    sentiment = c(
        "UMCSENT",        # University of Michigan: Consumer Sentiment
        "CSCICP03USM665S", # Consumer Confidence Index
        "USSLIND",        # Leading Index for the United States
        "PSAVERT",        # Personal Saving Rate
        "PCE",            # Personal Consumption Expenditures
        "PCECC96",        # Real Personal Consumption Expenditures
        "DSPI",           # Disposable Personal Income
        "DSPIC96",        # Real Disposable Personal Income
        "RSXFS",          # Advance Real Retail and Food Services Sales
        "PCECTPI",        # Personal Consumption Expenditures: Chain-type Price Index
        "PCEC96",         # Real Personal Consumption Expenditures
        "RECPROUSM156N"   # Smoothed U.S. Recession Probabilities
    ),

    # Financial Stress & Risk (18 indicators)
    risk = c(
        "VIXCLS",         # CBOE Volatility Index: VIX
        "BAMLH0A0HYM2",   # ICE BofA US High Yield Index Option-Adjusted Spread
        "STLFSI4",        # St. Louis Fed Financial Stress Index
        "TEDRATE",        # TED Spread
        "BAMLC0A0CM",     # ICE BofA US Corporate Index Option-Adjusted Spread
        "BAMLC0A4CBBB",   # ICE BofA BBB US Corporate Index Option-Adjusted Spread
        "BAMLH0A0HYM2EY", # ICE BofA US High Yield Index Effective Yield
        "DFII10",         # Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity
        "DFII5",          # Market Yield on U.S. Treasury Securities at 5-Year Constant Maturity
        "DFFLL",          # Upper Limit of Target Federal Funds Rate
        "DFEDTARU",       # Federal Funds Target Range - Upper Limit
        "DFEDTARL",       # Federal Funds Target Range - Lower Limit
        "RRPONTSYD",      # Overnight Reverse Repurchase Agreements: Treasury Securities
        "RPONTSYD",       # Repurchase Agreements: Treasury Securities Sold by the Federal Reserve
        "RRPONTTLD",      # Overnight Reverse Repurchase Agreements: Total Securities
        "WSHOSHO",        # Households and Nonprofit Organizations; Corporate Equities; Asset
        "BOGZ1FL893064105Q", # Rest of the world; corporate and foreign bonds; asset
        "NFCI"            # Chicago Fed National Financial Conditions Index
    ),

    # International & Trade (8 indicators)
    international = c(
        "DEXUSEU",        # U.S. / Euro Foreign Exchange Rate
        "DEXJPUS",        # Japan / U.S. Foreign Exchange Rate
        "DEXCHUS",        # China / U.S. Foreign Exchange Rate
        "DEXUSUK",        # U.S. / U.K. Foreign Exchange Rate
        "DEXCAUS",        # Canada / U.S. Foreign Exchange Rate
        "EXPGS",          # Exports of Goods and Services
        "IMPGS",          # Imports of Goods and Services
        "NETEXP"          # Net Exports of Goods and Services
    )
)

# Flatten the list for easier processing
all_fred_series <- unlist(fred_indicators, use.names = FALSE)
cat("Total FRED indicators to analyze:", length(all_fred_series), "\n")
print(head(all_fred_series, 10))

## 2. Data Collection & Preprocessing

In [ ]:
# Function to safely fetch FRED data
safe_fredr <- function(series_id, max_retries = 3) {
    for (i in 1:max_retries) {
        tryCatch({
            data <- fredr::fredr_series_observations(series_id = series_id) %>%
                dplyr::select(date, value) %>%
                dplyr::mutate(
                    series_id = series_id,
                    value = as.numeric(value)
                ) %>%
                filter(!is.na(value))
            return(data)
        }, error = function(e) {
            cat("Error fetching", series_id, "- attempt", i, ":", e$message, "\n")
            if (i == max_retries) {
                return(data.frame(date = as.Date(character()), value = numeric(), series_id = character()))
            }
            Sys.sleep(1)  # Wait before retry
        })
    }
}

# Fetch all FRED data
cat("Fetching FRED data...\n")
df_fred <- rbindlist(lapply(all_fred_series, safe_fredr))

# Summary of data collected
df_fred_summary <- df_fred %>%
    group_by(series_id) %>%
    summarise(
        first_date = min(date, na.rm = TRUE),
        last_date = max(date, na.rm = TRUE),
        observations = n(),
        .groups = 'drop'
    ) %>%
    arrange(desc(observations))

cat("\nData collection summary:\n")
cat("Total series successfully fetched:", nrow(df_fred_summary), "\n")
cat("Date range:", as.character(min(df_fred$date)), "to", as.character(max(df_fred$date)), "\n")
cat("Total observations:", nrow(df_fred), "\n")

# Display summary table
DT::datatable(df_fred_summary, 
              caption = "FRED Data Collection Summary",
              options = list(pageLength = 10, scrollX = TRUE))

## 3. Stock Market Data Integration

In [ ]:
# Define comprehensive stock universe with 100+ symbols
stock_symbols <- list(
    # Major Market Indices (15)
    indices = c(
        "^GSPC",   # S&P 500
        "^DJI",    # Dow Jones Industrial Average
        "^IXIC",   # NASDAQ Composite
        "^RUT",    # Russell 2000
        "^VIX",    # CBOE Volatility Index
        "^TNX",    # 10-Year Treasury Note Yield
        "^STOXX50E", # Euro Stoxx 50
        "^FTSE",   # FTSE 100
        "^N225",   # Nikkei 225
        "^HSI",    # Hang Seng Index
        "^AXJO",   # ASX All Ordinaries
        "^BVSP",   # Bovespa
        "^MXX",    # IPC Mexico
        "^KS11",   # KOSPI Composite Index
        "^TWII"    # Taiwan Weighted
    ),
    
    # Large Cap Technology Stocks (25)
    tech = c(
        "AAPL",    # Apple Inc.
        "MSFT",    # Microsoft Corporation
        "GOOGL",   # Alphabet Inc. Class A
        "GOOG",    # Alphabet Inc. Class C
        "AMZN",    # Amazon.com Inc.
        "NVDA",    # NVIDIA Corporation
        "TSLA",    # Tesla Inc.
        "META",    # Meta Platforms Inc.
        "NFLX",    # Netflix Inc.
        "CRM",     # Salesforce Inc.
        "ORCL",    # Oracle Corporation
        "ADBE",    # Adobe Inc.
        "INTC",    # Intel Corporation
        "AMD",     # Advanced Micro Devices Inc.
        "QCOM",    # QUALCOMM Incorporated
        "AVGO",    # Broadcom Inc.
        "TXN",     # Texas Instruments Incorporated
        "INTU",    # Intuit Inc.
        "CSCO",    # Cisco Systems Inc.
        "IBM",     # International Business Machines
        "NOW",     # ServiceNow Inc.
        "AMAT",    # Applied Materials Inc.
        "LRCX",    # Lam Research Corporation
        "KLAC",    # KLA Corporation
        "MRVL"     # Marvell Technology Inc.
    ),
    
    # Financial Sector Stocks (25)
    financial = c(
        "JPM",     # JPMorgan Chase & Co.
        "BAC",     # Bank of America Corporation
        "WFC",     # Wells Fargo & Company
        "GS",      # The Goldman Sachs Group Inc.
        "MS",      # Morgan Stanley
        "C",       # Citigroup Inc.
        "BLK",     # BlackRock Inc.
        "SCHW",    # The Charles Schwab Corporation
        "USB",     # U.S. Bancorp
        "PNC",     # The PNC Financial Services Group
        "TFC",     # Truist Financial Corporation
        "COF",     # Capital One Financial Corporation
        "AXP",     # American Express Company
        "SPG",     # Simon Property Group Inc.
        "BX",      # Blackstone Inc.
        "KKR",     # KKR & Co. Inc.
        "APO",     # Apollo Global Management Inc.
        "V",       # Visa Inc.
        "MA",      # Mastercard Incorporated
        "PYPL",    # PayPal Holdings Inc.
        "FIS",     # Fidelity National Information Services
        "FISV",    # Fiserv Inc.
        "AIG",     # American International Group Inc.
        "PRU",     # Prudential Financial Inc.
        "MET"      # MetLife Inc.
    ),
    
    # Healthcare & Pharmaceuticals (20)
    healthcare = c(
        "JNJ",     # Johnson & Johnson
        "UNH",     # UnitedHealth Group Incorporated
        "PFE",     # Pfizer Inc.
        "ABBV",    # AbbVie Inc.
        "TMO",     # Thermo Fisher Scientific Inc.
        "ABT",     # Abbott Laboratories
        "DHR",     # Danaher Corporation
        "MRK",     # Merck & Co. Inc.
        "BMY",     # Bristol-Myers Squibb Company
        "AMGN",    # Amgen Inc.
        "GILD",    # Gilead Sciences Inc.
        "CVS",     # CVS Health Corporation
        "ANTM",    # Anthem Inc.
        "CI",      # Cigna Corporation
        "HUM",     # Humana Inc.
        "ISRG",    # Intuitive Surgical Inc.
        "REGN",    # Regeneron Pharmaceuticals Inc.
        "VRTX",    # Vertex Pharmaceuticals Incorporated
        "BIIB",    # Biogen Inc.
        "MRNA"     # Moderna Inc.
    ),
    
    # Consumer Discretionary (20)
    consumer_disc = c(
        "AMZN",    # Amazon.com Inc. (also in tech)
        "TSLA",    # Tesla Inc. (also in tech)
        "HD",      # The Home Depot Inc.
        "MCD",     # McDonald's Corporation
        "NKE",     # NIKE Inc.
        "SBUX",    # Starbucks Corporation
        "LOW",     # Lowe's Companies Inc.
        "TJX",     # The TJX Companies Inc.
        "BKNG",    # Booking Holdings Inc.
        "ABNB",    # Airbnb Inc.
        "UBER",    # Uber Technologies Inc.
        "LYFT",    # Lyft Inc.
        "F",       # Ford Motor Company
        "GM",      # General Motors Company
        "DIS",     # The Walt Disney Company
        "NFLX",    # Netflix Inc. (also in tech)
        "CMCSA",   # Comcast Corporation
        "VZ",      # Verizon Communications Inc.
        "T",       # AT&T Inc.
        "TMUS"     # T-Mobile US Inc.
    ),
    
    # Consumer Staples (15)
    consumer_staples = c(
        "PG",      # The Procter & Gamble Company
        "KO",      # The Coca-Cola Company
        "PEP",     # PepsiCo Inc.
        "WMT",     # Walmart Inc.
        "COST",    # Costco Wholesale Corporation
        "CL",      # Colgate-Palmolive Company
        "KMB",     # Kimberly-Clark Corporation
        "GIS",     # General Mills Inc.
        "K",       # Kellogg Company
        "HSY",     # The Hershey Company
        "MDLZ",    # Mondelez International Inc.
        "KHC",     # The Kraft Heinz Company
        "CPB",     # Campbell Soup Company
        "SJM",     # The J. M. Smucker Company
        "CAG"      # Conagra Brands Inc.
    ),
    
    # Industrial & Materials (15)
    industrial = c(
        "CAT",     # Caterpillar Inc.
        "BA",      # The Boeing Company
        "GE",      # General Electric Company
        "MMM",     # 3M Company
        "HON",     # Honeywell International Inc.
        "UPS",     # United Parcel Service Inc.
        "FDX",     # FedEx Corporation
        "LMT",     # Lockheed Martin Corporation
        "RTX",     # Raytheon Technologies Corporation
        "NOC",     # Northrop Grumman Corporation
        "DE",      # Deere & Company
        "EMR",     # Emerson Electric Co.
        "ITW",     # Illinois Tool Works Inc.
        "ETN",     # Eaton Corporation plc
        "PH"       # Parker-Hannifin Corporation
    ),
    
    # Energy Sector (12)
    energy = c(
        "XOM",     # Exxon Mobil Corporation
        "CVX",     # Chevron Corporation
        "COP",     # ConocoPhillips
        "EOG",     # EOG Resources Inc.
        "SLB",     # Schlumberger Limited
        "PXD",     # Pioneer Natural Resources Company
        "KMI",     # Kinder Morgan Inc.
        "OKE",     # ONEOK Inc.
        "WMB",     # The Williams Companies Inc.
        "MPC",     # Marathon Petroleum Corporation
        "VLO",     # Valero Energy Corporation
        "PSX"      # Phillips 66
    ),
    
    # Utilities (10)
    utilities = c(
        "NEE",     # NextEra Energy Inc.
        "DUK",     # Duke Energy Corporation
        "SO",      # The Southern Company
        "D",       # Dominion Energy Inc.
        "AEP",     # American Electric Power Company Inc.
        "EXC",     # Exelon Corporation
        "XEL",     # Xcel Energy Inc.
        "SRE",     # Sempra Energy
        "PEG",     # Public Service Enterprise Group
        "ED"       # Consolidated Edison Inc.
    ),
    
    # Real Estate & REITs (10)
    real_estate = c(
        "AMT",     # American Tower Corporation
        "PLD",     # Prologis Inc.
        "CCI",     # Crown Castle International Corp.
        "EQIX",    # Equinix Inc.
        "WELL",    # Welltower Inc.
        "DLR",     # Digital Realty Trust Inc.
        "SBAC",    # SBA Communications Corporation
        "O",       # Realty Income Corporation
        "VICI",    # VICI Properties Inc.
        "AVB"      # AvalonBay Communities Inc.
    ),
    
    # Other Large Cap (Berkshire, etc.) (8)
    other = c(
        "BRK-A",   # Berkshire Hathaway Inc. Class A
        "BRK-B",   # Berkshire Hathaway Inc. Class B
        "LLY",     # Eli Lilly and Company
        "AVGO",    # Broadcom Inc. (also in tech)
        "WMT",     # Walmart Inc. (also in consumer staples)
        "V",       # Visa Inc. (also in financial)
        "JPM",     # JPMorgan Chase & Co. (also in financial)
        "UNH"      # UnitedHealth Group (also in healthcare)
    )
)

# Remove duplicates and flatten
all_stock_symbols <- unique(unlist(stock_symbols, use.names = FALSE))
cat("Total unique stock symbols to analyze:", length(all_stock_symbols), "\n")

# Function to safely fetch stock data with better error handling
safe_getSymbols <- function(symbol, max_retries = 3) {
    for (i in 1:max_retries) {
        tryCatch({
            # Some symbols might need special handling
            clean_symbol <- gsub("\\^", "", symbol)  # Remove ^ for some APIs
            
            data <- quantmod::getSymbols(symbol, 
                                       env = NULL, 
                                       auto.assign = FALSE,
                                       from = Sys.Date() - years(15)) %>%
                as.data.frame() %>%
                mutate(date = as.Date(row.names(.))) %>%
                select(date, 4) %>%  # Select Close price (4th column)
                setNames(c("date", "value")) %>%
                mutate(
                    series_id = symbol,
                    value = as.numeric(value)
                ) %>%
                filter(!is.na(value), is.finite(value))
            
            rownames(data) <- NULL
            return(data)
        }, error = function(e) {
            cat("Error fetching", symbol, "- attempt", i, ":", e$message, "\n")
            if (i == max_retries) {
                return(data.frame(date = as.Date(character()), value = numeric(), series_id = character()))
            }
            Sys.sleep(2)  # Longer wait to avoid rate limits
        })
    }
}

# Fetch stock data
cat("Fetching comprehensive stock market data...\n")
df_stocks <- rbindlist(lapply(all_stock_symbols, safe_getSymbols))

# Summary of stock data
df_stocks_summary <- df_stocks %>%
    group_by(series_id) %>%
    summarise(
        first_date = min(date, na.rm = TRUE),
        last_date = max(date, na.rm = TRUE),
        observations = n(),
        .groups = 'drop'
    ) %>%
    arrange(desc(observations))

cat("\nStock data collection summary:\n")
cat("Total stocks successfully fetched:", nrow(df_stocks_summary), "\n")
cat("Total stock observations:", nrow(df_stocks), "\n")

DT::datatable(df_stocks_summary, 
              caption = "Comprehensive Stock Market Data Collection Summary",
              options = list(pageLength = 15, scrollX = TRUE))

## 4. Data Harmonization & Alignment

In [ ]:
# Combine all data sources
df_all_data <- bind_rows(
    df_fred %>% mutate(data_source = "FRED"),
    df_stocks %>% mutate(data_source = "Stock")
)

# Create a common date range (last 10 years for better data availability)
end_date <- Sys.Date()
start_date <- end_date - years(10)

# Filter to common date range and ensure data quality
df_filtered <- df_all_data %>%
    filter(
        date >= start_date,
        date <= end_date,
        !is.na(value),
        is.finite(value)
    ) %>%
    group_by(series_id) %>%
    # Only keep series with at least 500 observations (roughly 2 years of data)
    filter(n() >= 500) %>%
    ungroup()

# Get list of series with sufficient data
valid_series <- df_filtered %>%
    group_by(series_id, data_source) %>%
    summarise(
        observations = n(),
        date_range = paste(min(date), "to", max(date)),
        .groups = 'drop'
    ) %>%
    arrange(data_source, desc(observations))

cat("\nData after filtering (last 10 years, min 500 observations):\n")
cat("FRED series:", sum(valid_series$data_source == "FRED"), "\n")
cat("Stock series:", sum(valid_series$data_source == "Stock"), "\n")
cat("Total valid series:", nrow(valid_series), "\n")

DT::datatable(valid_series, 
              caption = "Valid Series for Analysis",
              options = list(pageLength = 15, scrollX = TRUE))

## 5. Correlation Analysis Framework

In [ ]:
# Function to calculate cross-correlation with lags
calculate_cross_correlation <- function(data, max_lag = 60) {
    # Get all unique series
    series_list <- unique(data$series_id)
    
    # Create all pairwise combinations
    combinations <- expand.grid(
        x1 = series_list,
        x2 = series_list,
        stringsAsFactors = FALSE
    ) %>%
    filter(x1 != x2)  # Remove self-correlations
    
    cat("Calculating correlations for", nrow(combinations), "pairs...\n")
    
    # Function to calculate CCF for a single pair
    calc_pair_ccf <- function(x1, x2, data, max_lag) {
        tryCatch({
            # Get data for both series on same dates
            data1 <- data %>% filter(series_id == x1) %>% select(date, value)
            data2 <- data %>% filter(series_id == x2) %>% select(date, value)
            
            # Merge on common dates
            merged <- inner_join(data1, data2, by = "date", suffix = c(".x", ".y")) %>%
                arrange(date) %>%
                filter(
                    !is.na(value.x), !is.na(value.y),
                    is.finite(value.x), is.finite(value.y)
                )
            
            if (nrow(merged) < 100) {
                return(NULL)  # Not enough overlapping data
            }
            
            # Calculate cross-correlation
            ccf_result <- ccf(merged$value.x, merged$value.y, 
                            lag.max = min(max_lag, floor(nrow(merged)/4)), 
                            plot = FALSE)
            
            # Find best correlation
            best_idx <- which.max(abs(ccf_result$acf))
            best_lag <- ccf_result$lag[best_idx]
            best_corr <- ccf_result$acf[best_idx]
            
            # Also get zero-lag correlation
            zero_lag_idx <- which(ccf_result$lag == 0)
            zero_lag_corr <- ifelse(length(zero_lag_idx) > 0, ccf_result$acf[zero_lag_idx], NA)
            
            return(data.frame(
                x1 = x1,
                x2 = x2,
                best_lag = best_lag,
                best_corr = best_corr,
                zero_lag_corr = zero_lag_corr,
                n_obs = nrow(merged),
                stringsAsFactors = FALSE
            ))
        }, error = function(e) {
            return(NULL)
        })
    }
    
    # Calculate correlations in batches to avoid memory issues
    batch_size <- 1000
    n_batches <- ceiling(nrow(combinations) / batch_size)
    
    correlation_results <- list()
    
    for (i in 1:n_batches) {
        start_idx <- (i - 1) * batch_size + 1
        end_idx <- min(i * batch_size, nrow(combinations))
        
        cat("Processing batch", i, "of", n_batches, "\n")
        
        batch_combinations <- combinations[start_idx:end_idx, ]
        
        batch_results <- mapply(
            calc_pair_ccf,
            batch_combinations$x1,
            batch_combinations$x2,
            MoreArgs = list(data = data, max_lag = max_lag),
            SIMPLIFY = FALSE
        )
        
        # Remove NULL results
        batch_results <- batch_results[!sapply(batch_results, is.null)]
        
        if (length(batch_results) > 0) {
            correlation_results[[i]] <- rbindlist(batch_results)
        }
    }
    
    # Combine all results
    if (length(correlation_results) > 0) {
        final_results <- rbindlist(correlation_results)
        return(final_results)
    } else {
        return(data.frame())
    }
}

# Calculate correlations
cat("Starting correlation analysis...\n")
df_correlations <- calculate_cross_correlation(df_filtered, max_lag = 30)

if (nrow(df_correlations) > 0) {
    cat("\nCorrelation analysis complete!\n")
    cat("Total correlations calculated:", nrow(df_correlations), "\n")
    cat("Average absolute correlation:", round(mean(abs(df_correlations$best_corr), na.rm = TRUE), 3), "\n")
    
    # Show top correlations
    top_correlations <- df_correlations %>%
        arrange(desc(abs(best_corr))) %>%
        head(20) %>%
        mutate(
            best_corr = round(best_corr, 3),
            zero_lag_corr = round(zero_lag_corr, 3)
        )
    
    DT::datatable(top_correlations, 
                  caption = "Top 20 Correlations by Absolute Value",
                  options = list(pageLength = 20, scrollX = TRUE))
} else {
    cat("No correlations calculated. Check data availability.\n")
}

## 6. Knowledge Graph Construction

In [ ]:
# Create knowledge graph based on strong correlations
if (nrow(df_correlations) > 0) {
    
    # Filter for significant correlations
    significance_threshold <- 0.7
    
    strong_correlations <- df_correlations %>%
        filter(
            abs(best_corr) >= significance_threshold,
            n_obs >= 500  # Ensure statistical significance
        ) %>%
        mutate(
            correlation_type = case_when(
                best_corr > 0 ~ "positive",
                best_corr < 0 ~ "negative",
                TRUE ~ "neutral"
            ),
            lag_type = case_when(
                best_lag < -5 ~ "leading",
                best_lag > 5 ~ "lagging", 
                TRUE ~ "coincident"
            ),
            strength = abs(best_corr)
        )
    
    cat("Strong correlations (|r| >=", significance_threshold, "):", nrow(strong_correlations), "\n")
    
    if (nrow(strong_correlations) > 0) {
        
        # Create nodes data frame
        all_nodes <- unique(c(strong_correlations$x1, strong_correlations$x2))
        
        # Add node metadata
        nodes_df <- data.frame(
            id = all_nodes,
            label = all_nodes,
            stringsAsFactors = FALSE
        ) %>%
        mutate(
            # Classify node types
            node_type = case_when(
                id %in% names(fred_indicators$employment) ~ "Employment",
                id %in% names(fred_indicators$inflation) ~ "Inflation",
                id %in% names(fred_indicators$rates) ~ "Interest Rates",
                id %in% names(fred_indicators$money) ~ "Money Supply",
                id %in% names(fred_indicators$output) ~ "Economic Output",
                id %in% names(fred_indicators$housing) ~ "Housing",
                id %in% names(fred_indicators$sentiment) ~ "Sentiment",
                id %in% names(fred_indicators$risk) ~ "Financial Risk",
                id %in% unlist(stock_symbols$indices) ~ "Market Index",
                id %in% unlist(stock_symbols$tech) ~ "Tech Stock",
                id %in% unlist(stock_symbols$financial) ~ "Financial Stock",
                id %in% unlist(stock_symbols$other) ~ "Other Stock",
                TRUE ~ "Other"
            ),
            # Color by type
            color = case_when(
                node_type == "Employment" ~ "#1f77b4",
                node_type == "Inflation" ~ "#ff7f0e", 
                node_type == "Interest Rates" ~ "#2ca02c",
                node_type == "Money Supply" ~ "#d62728",
                node_type == "Economic Output" ~ "#9467bd",
                node_type == "Housing" ~ "#8c564b",
                node_type == "Sentiment" ~ "#e377c2",
                node_type == "Financial Risk" ~ "#7f7f7f",
                node_type == "Market Index" ~ "#bcbd22",
                node_type == "Tech Stock" ~ "#17becf",
                node_type == "Financial Stock" ~ "#ff9896",
                node_type == "Other Stock" ~ "#c5b0d5",
                TRUE ~ "#aec7e8"
            )
        )
        
        # Calculate node importance (centrality)
        node_importance <- strong_correlations %>%
            select(x1, x2, strength) %>%
            pivot_longer(cols = c(x1, x2), names_to = "position", values_to = "node") %>%
            group_by(node) %>%
            summarise(
                degree = n(),  # Number of connections
                avg_strength = mean(strength, na.rm = TRUE),
                total_strength = sum(strength, na.rm = TRUE),
                .groups = 'drop'
            )
        
        # Add importance to nodes
        nodes_df <- nodes_df %>%
            left_join(node_importance, by = c("id" = "node")) %>%
            mutate(
                degree = ifelse(is.na(degree), 0, degree),
                avg_strength = ifelse(is.na(avg_strength), 0, avg_strength),
                total_strength = ifelse(is.na(total_strength), 0, total_strength),
                size = scales::rescale(degree, to = c(10, 50)),  # Size based on connections
                title = paste0(
                    "<b>", label, "</b><br>",
                    "Type: ", node_type, "<br>",
                    "Degree: ", degree, "<br>",
                    "Avg Correlation: ", round(avg_strength, 3)
                )
            )
        
        # Create edges data frame
        edges_df <- strong_correlations %>%
            mutate(
                from = x1,
                to = x2,
                weight = strength,
                width = scales::rescale(strength, to = c(1, 8)),
                color = ifelse(correlation_type == "positive", "#2ca02c", "#d62728"),
                title = paste0(
                    "<b>", x1, " → ", x2, "</b><br>",
                    "Correlation: ", round(best_corr, 3), "<br>",
                    "Lag: ", best_lag, " days<br>",
                    "Type: ", lag_type, "<br>",
                    "Observations: ", n_obs
                ),
                arrows = "to"
            ) %>%
            select(from, to, weight, width, color, title, arrows)
        
        cat("Knowledge graph created:\n")
        cat("Nodes:", nrow(nodes_df), "\n")
        cat("Edges:", nrow(edges_df), "\n")
        
        # Display top influential nodes
        top_nodes <- nodes_df %>%
            arrange(desc(degree)) %>%
            head(15) %>%
            select(label, node_type, degree, avg_strength) %>%
            mutate(avg_strength = round(avg_strength, 3))
        
        DT::datatable(top_nodes, 
                      caption = "Most Influential Nodes (Highest Degree Centrality)",
                      options = list(pageLength = 15))
        
    } else {
        cat("No strong correlations found with threshold", significance_threshold, "\n")
    }
} else {
    cat("Cannot create knowledge graph - no correlation data available\n")
}

## 7. Interactive Knowledge Graph Visualization

## 8. Correlation Heatmap Analysis

In [ ]:
# Create correlation matrix for heatmap
if (nrow(df_correlations) > 0) {
    
    # Select most connected indicators for readability
    top_indicators <- node_importance %>%
        arrange(desc(degree)) %>%
        head(25) %>%
        pull(node)
    
    # Create correlation matrix
    correlation_matrix <- df_correlations %>%
        filter(x1 %in% top_indicators, x2 %in% top_indicators) %>%
        select(x1, x2, zero_lag_corr) %>%
        pivot_wider(names_from = x2, values_from = zero_lag_corr, values_fill = NA) %>%
        column_to_rownames("x1") %>%
        as.matrix()
    
    # Make matrix symmetric
    for (i in 1:nrow(correlation_matrix)) {
        for (j in 1:ncol(correlation_matrix)) {
            if (is.na(correlation_matrix[i, j]) && !is.na(correlation_matrix[j, i])) {
                correlation_matrix[i, j] <- correlation_matrix[j, i]
            }
        }
    }
    
    # Set diagonal to 1
    diag(correlation_matrix) <- 1
    
    # Create heatmap plot
    if (nrow(correlation_matrix) > 1 && ncol(correlation_matrix) > 1) {
        
        # Convert to long format for ggplot
        heatmap_data <- correlation_matrix %>%
            as.data.frame() %>%
            rownames_to_column("indicator1") %>%
            pivot_longer(-indicator1, names_to = "indicator2", values_to = "correlation") %>%
            filter(!is.na(correlation))
        
        # Create ggplot heatmap
        heatmap_plot <- ggplot(heatmap_data, aes(x = indicator1, y = indicator2, fill = correlation)) +
            geom_tile(color = "white", size = 0.1) +
            scale_fill_gradient2(
                low = "#d62728", mid = "white", high = "#2ca02c",
                midpoint = 0, limit = c(-1, 1), space = "Lab",
                name = "Correlation"
            ) +
            theme_minimal() +
            theme(
                axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
                axis.text.y = element_text(size = 8),
                axis.title = element_blank(),
                panel.grid = element_blank(),
                legend.position = "right"
            ) +
            coord_fixed() +
            labs(
                title = "Economic Indicators Correlation Heatmap",
                subtitle = paste("Top", length(top_indicators), "Most Connected Indicators"),
                caption = "Data: FRED API & Yahoo Finance"
            )
        
        print(heatmap_plot)
        
        # Also create an interactive plotly version
        interactive_heatmap <- plot_ly(
            z = ~correlation_matrix,
            type = "heatmap",
            colors = c("red", "white", "green"),
            hovertemplate = "<b>%{x} vs %{y}</b><br>Correlation: %{z:.3f}<extra></extra>"
        ) %>%
        layout(
            title = "Interactive Correlation Heatmap",
            xaxis = list(title = "", tickangle = 45),
            yaxis = list(title = "")
        )
        
        interactive_heatmap
        
    } else {
        cat("Insufficient data for correlation matrix\n")
    }
} else {
    cat("No correlation data available for heatmap\n")
}

## 9. Economic Driver Analysis

In [ ]:
# Analyze which indicators are universal drivers vs followers
if (exists("strong_correlations") && nrow(strong_correlations) > 0) {
    
    # Calculate leadership scores based on lag patterns
    leadership_analysis <- strong_correlations %>%
        group_by(x1) %>%
        summarise(
            total_connections = n(),
            avg_lag = mean(best_lag, na.rm = TRUE),
            median_lag = median(best_lag, na.rm = TRUE),
            leading_connections = sum(best_lag < -5, na.rm = TRUE),
            coincident_connections = sum(abs(best_lag) <= 5, na.rm = TRUE),
            lagging_connections = sum(best_lag > 5, na.rm = TRUE),
            avg_correlation_strength = mean(abs(best_corr), na.rm = TRUE),
            .groups = 'drop'
        ) %>%
        mutate(
            # Calculate leadership score
            leadership_score = (leading_connections - lagging_connections) / total_connections,
            # Classify indicator type
            indicator_role = case_when(
                leadership_score > 0.3 ~ "Strong Leader",
                leadership_score > 0.1 ~ "Moderate Leader", 
                leadership_score > -0.1 ~ "Coincident",
                leadership_score > -0.3 ~ "Moderate Follower",
                TRUE ~ "Strong Follower"
            )
        ) %>%
        arrange(desc(leadership_score))
    
    # Add indicator metadata
    leadership_analysis <- leadership_analysis %>%
        mutate(
            indicator_category = case_when(
                x1 %in% unlist(fred_indicators$employment) ~ "Employment",
                x1 %in% unlist(fred_indicators$inflation) ~ "Inflation",
                x1 %in% unlist(fred_indicators$rates) ~ "Interest Rates",
                x1 %in% unlist(fred_indicators$money) ~ "Money Supply",
                x1 %in% unlist(fred_indicators$output) ~ "Economic Output",
                x1 %in% unlist(fred_indicators$housing) ~ "Housing",
                x1 %in% unlist(fred_indicators$sentiment) ~ "Sentiment",
                x1 %in% unlist(fred_indicators$risk) ~ "Financial Risk",
                x1 %in% unlist(stock_symbols$indices) ~ "Market Index",
                x1 %in% unlist(stock_symbols) ~ "Stock",
                TRUE ~ "Other"
            )
        )
    
    cat("\nEconomic Driver Analysis Results:\n")
    cat("Total indicators analyzed:", nrow(leadership_analysis), "\n")
    
    # Display leadership rankings
    leadership_display <- leadership_analysis %>%
        select(
            Indicator = x1,
            Category = indicator_category,
            Role = indicator_role,
            `Leadership Score` = leadership_score,
            `Total Connections` = total_connections,
            `Leading Connections` = leading_connections,
            `Lagging Connections` = lagging_connections,
            `Avg Correlation` = avg_correlation_strength
        ) %>%
        mutate(
            `Leadership Score` = round(`Leadership Score`, 3),
            `Avg Correlation` = round(`Avg Correlation`, 3)
        )
    
    DT::datatable(leadership_display, 
                  caption = "Economic Indicators: Leadership Analysis",
                  options = list(pageLength = 20, scrollX = TRUE)) %>%
        formatStyle(
            "Leadership Score",
            background = styleColorBar(range(leadership_display$`Leadership Score`), "lightblue"),
            backgroundSize = "100% 90%",
            backgroundRepeat = "no-repeat",
            backgroundPosition = "center"
        )
    
} else {
    cat("No strong correlations available for driver analysis\n")
}

## 10. Stock Market Integration Analysis

In [ ]:
# Analyze relationships between economic indicators and stock market
if (exists("strong_correlations") && nrow(strong_correlations) > 0) {
    
    # Focus on stock market relationships
    stock_relationships <- strong_correlations %>%
        filter(
            x1 %in% unlist(stock_symbols) | x2 %in% unlist(stock_symbols)
        ) %>%
        mutate(
            stock_symbol = ifelse(x1 %in% unlist(stock_symbols), x1, x2),
            economic_indicator = ifelse(x1 %in% unlist(stock_symbols), x2, x1),
            stock_leads = ifelse(
                x1 %in% unlist(stock_symbols),
                best_lag < 0,  # x1 leads x2
                best_lag > 0   # x2 leads x1 (stock is x2)
            )
        ) %>%
        # Add stock categorization
        mutate(
            stock_type = case_when(
                stock_symbol %in% unlist(stock_symbols$indices) ~ "Market Index",
                stock_symbol %in% unlist(stock_symbols$tech) ~ "Tech Stock",
                stock_symbol %in% unlist(stock_symbols$financial) ~ "Financial Stock",
                stock_symbol %in% unlist(stock_symbols$other) ~ "Other Large Cap",
                TRUE ~ "Unknown"
            ),
            # Add economic indicator categorization
            econ_category = case_when(
                economic_indicator %in% unlist(fred_indicators$employment) ~ "Employment",
                economic_indicator %in% unlist(fred_indicators$inflation) ~ "Inflation",
                economic_indicator %in% unlist(fred_indicators$rates) ~ "Interest Rates",
                economic_indicator %in% unlist(fred_indicators$money) ~ "Money Supply",
                economic_indicator %in% unlist(fred_indicators$output) ~ "Economic Output",
                economic_indicator %in% unlist(fred_indicators$housing) ~ "Housing",
                economic_indicator %in% unlist(fred_indicators$sentiment) ~ "Sentiment",
                economic_indicator %in% unlist(fred_indicators$risk) ~ "Financial Risk",
                TRUE ~ "Other"
            )
        )
    
    if (nrow(stock_relationships) > 0) {
        
        cat("\nStock Market Integration Analysis:\n")
        cat("Total stock-economic relationships:", nrow(stock_relationships), "\n")
        
        # Summary by stock type and economic category
        relationship_summary <- stock_relationships %>%
            group_by(stock_type, econ_category) %>%
            summarise(
                count = n(),
                avg_correlation = mean(abs(best_corr), na.rm = TRUE),
                avg_lag = mean(abs(best_lag), na.rm = TRUE),
                stock_leads_pct = mean(stock_leads, na.rm = TRUE) * 100,
                .groups = 'drop'
            ) %>%
            arrange(desc(count)) %>%
            mutate(
                avg_correlation = round(avg_correlation, 3),
                avg_lag = round(avg_lag, 1),
                stock_leads_pct = round(stock_leads_pct, 1)
            )
        
        DT::datatable(relationship_summary, 
                      caption = "Stock Market - Economic Indicator Relationships",
                      colnames = c(
                          "Stock Type", "Economic Category", "Relationships", 
                          "Avg |Correlation|", "Avg |Lag| (days)", "Stock Leads (%)"
                      ),
                      options = list(pageLength = 15, scrollX = TRUE))
        
        # Most significant stock-economic relationships
        top_stock_relationships <- stock_relationships %>%
            arrange(desc(abs(best_corr))) %>%
            head(20) %>%
            select(
                `Stock Symbol` = stock_symbol,
                `Stock Type` = stock_type,
                `Economic Indicator` = economic_indicator,
                `Econ Category` = econ_category,
                `Correlation` = best_corr,
                `Lag (days)` = best_lag,
                `Stock Leads` = stock_leads,
                `Observations` = n_obs
            ) %>%
            mutate(
                Correlation = round(Correlation, 3)
            )
        
        DT::datatable(top_stock_relationships, 
                      caption = "Top 20 Stock-Economic Indicator Correlations",
                      options = list(pageLength = 20, scrollX = TRUE))
        
        # Create visualization of stock-economic relationships
        if (nrow(relationship_summary) > 0) {
            
            relationship_plot <- ggplot(relationship_summary, 
                                      aes(x = avg_correlation, y = stock_leads_pct, 
                                          size = count, color = stock_type)) +
                geom_point(alpha = 0.7) +
                scale_size_continuous(range = c(3, 15), name = "# Relationships") +
                scale_color_brewer(type = "qual", palette = "Set1", name = "Stock Type") +
                labs(
                    title = "Stock Market Leadership vs Economic Correlation",
                    subtitle = "Each point represents a stock type - economic category combination",
                    x = "Average |Correlation| with Economic Indicators",
                    y = "% of Time Stock Leads Economic Indicator",
                    caption = "Size = Number of relationships"
                ) +
                theme_minimal() +
                theme(legend.position = "right") +
                geom_hline(yintercept = 50, linetype = "dashed", alpha = 0.5) +
                geom_vline(xintercept = 0.7, linetype = "dashed", alpha = 0.5)
            
            print(relationship_plot)
        }
        
    } else {
        cat("No significant stock-economic indicator relationships found\n")
    }
} else {
    cat("No correlation data available for stock market analysis\n")
}

## 11. Dynamic Knowledge Graph Export

In [ ]:
# Export knowledge graph data for further analysis
if (exists("nodes_df") && exists("edges_df") && exists("leadership_analysis")) {
    
    # Create comprehensive export data
    export_data <- list(
        metadata = list(
            created_date = Sys.Date(),
            analysis_period = paste(start_date, "to", end_date),
            total_indicators = length(unique(df_filtered$series_id)),
            correlation_threshold = significance_threshold,
            fred_indicators = fred_indicators,
            stock_symbols = stock_symbols
        ),
        nodes = nodes_df,
        edges = edges_df,
        correlations = if(exists("strong_correlations")) strong_correlations else NULL,
        leadership = leadership_analysis,
        stock_relationships = if(exists("stock_relationships")) stock_relationships else NULL
    )
    
    # Save as RDS for R users
    saveRDS(export_data, "knowledgegraph/knowledge_graph_data.rds")
    
    # Save individual components as CSV for broader accessibility
    write.csv(nodes_df, "knowledgegraph/knowledge_graph_nodes.csv", row.names = FALSE)
    write.csv(edges_df, "knowledgegraph/knowledge_graph_edges.csv", row.names = FALSE)
    write.csv(leadership_analysis, "knowledgegraph/leadership_analysis.csv", row.names = FALSE)
    
    if (exists("strong_correlations")) {
        write.csv(strong_correlations, "knowledgegraph/correlations_detailed.csv", row.names = FALSE)
    }
    
    if (exists("stock_relationships")) {
        write.csv(stock_relationships, "knowledgegraph/stock_economic_relationships.csv", row.names = FALSE)
    }
    
    cat("\nKnowledge Graph Export Complete!\n")
    cat("Files saved to knowledgegraph/ directory:\n")
    cat("- knowledge_graph_data.rds (complete R data)\n")
    cat("- knowledge_graph_nodes.csv\n")
    cat("- knowledge_graph_edges.csv\n")
    cat("- leadership_analysis.csv\n")
    cat("- correlations_detailed.csv\n")
    cat("- stock_economic_relationships.csv\n")
    
} else {
    cat("Insufficient data for export\n")
}

## 12. Summary & Key Insights

In [ ]:
# Generate comprehensive summary
cat("\n" , "=" %>% rep(80) %>% paste(collapse=""), "\n")
cat("ECONOMIC INDICATORS KNOWLEDGE GRAPH - ANALYSIS SUMMARY\n")
cat("=" %>% rep(80) %>% paste(collapse=""), "\n\n")

# Data Summary
cat("DATA COLLECTION:\n")
cat("- Analysis Period:", as.character(start_date), "to", as.character(end_date), "\n")
cat("- FRED Indicators Analyzed:", sum(valid_series$data_source == "FRED"), "\n")
cat("- Stock Symbols Analyzed:", sum(valid_series$data_source == "Stock"), "\n")
cat("- Total Data Points:", scales::comma(nrow(df_filtered)), "\n\n")

if (exists("strong_correlations") && nrow(strong_correlations) > 0) {
    # Correlation Analysis Summary
    cat("CORRELATION ANALYSIS:\n")
    cat("- Correlation Threshold:", significance_threshold, "\n")
    cat("- Strong Correlations Found:", scales::comma(nrow(strong_correlations)), "\n")
    cat("- Average Correlation Strength:", round(mean(abs(strong_correlations$best_corr)), 3), "\n")
    cat("- Positive Correlations:", scales::comma(sum(strong_correlations$correlation_type == "positive")), "\n")
    cat("- Negative Correlations:", scales::comma(sum(strong_correlations$correlation_type == "negative")), "\n\n")
}

if (exists("nodes_df") && exists("edges_df")) {
    # Network Analysis Summary
    cat("KNOWLEDGE GRAPH STRUCTURE:\n")
    cat("- Network Nodes:", nrow(nodes_df), "\n")
    cat("- Network Edges:", nrow(edges_df), "\n")
    cat("- Average Node Degree:", round(mean(nodes_df$degree, na.rm = TRUE), 1), "\n")
    cat("- Most Connected Indicator:", nodes_df$label[order(nodes_df$degree, decreasing = TRUE)[1]], 
        "(", max(nodes_df$degree, na.rm = TRUE), "connections)\n\n")
}

if (exists("leadership_analysis") && nrow(leadership_analysis) > 0) {
    # Leadership Analysis Summary
    cat("ECONOMIC DRIVERS ANALYSIS:\n")
    
    top_leaders <- leadership_analysis %>%
        filter(indicator_role %in% c("Strong Leader", "Moderate Leader")) %>%
        head(5)
    
    if (nrow(top_leaders) > 0) {
        cat("- Top Economic Drivers:\n")
        for (i in 1:nrow(top_leaders)) {
            cat("  ", i, ". ", top_leaders$x1[i], " (", top_leaders$indicator_category[i], ") - ",
                "Score: ", round(top_leaders$leadership_score[i], 3), "\n")
        }
        cat("\n")
    }
    
    role_summary <- table(leadership_analysis$indicator_role)
    cat("- Indicator Roles Distribution:\n")
    for (role in names(role_summary)) {
        cat("  ", role, ":", role_summary[role], "\n")
    }
    cat("\n")
}

if (exists("stock_relationships") && nrow(stock_relationships) > 0) {
    # Stock Market Integration Summary
    cat("STOCK MARKET INTEGRATION:\n")
    cat("- Stock-Economic Relationships:", nrow(stock_relationships), "\n")
    cat("- Average Stock-Economic Correlation:", 
        round(mean(abs(stock_relationships$best_corr), na.rm = TRUE), 3), "\n")
    
    stock_leads_pct <- mean(stock_relationships$stock_leads, na.rm = TRUE) * 100
    cat("- Stock Leads Economic Indicators:", round(stock_leads_pct, 1), "% of time\n")
    
    # Top stock-economic correlations
    top_stock_corr <- stock_relationships %>%
        arrange(desc(abs(best_corr))) %>%
        head(3)
    
    if (nrow(top_stock_corr) > 0) {
        cat("- Strongest Stock-Economic Correlations:\n")
        for (i in 1:nrow(top_stock_corr)) {
            cat("  ", i, ". ", top_stock_corr$stock_symbol[i], " ↔ ", top_stock_corr$economic_indicator[i],
                " (r=", round(top_stock_corr$best_corr[i], 3), ")\n")
        }
    }
    cat("\n")
}
# Key Insights
cat("KEY INSIGHTS:\n")
cat("1. The economic indicator network reveals complex interdependencies\n")
cat("   between monetary policy, employment, inflation, and market performance.\n\n")

cat("2. Leading indicators tend to be monetary policy tools (interest rates,\n")
cat("   money supply) and financial stress measures.\n\n")

cat("3. Stock market indices show significant correlations with economic\n")
cat("   fundamentals, with varying lead-lag relationships.\n\n")

cat("4. The knowledge graph structure can help identify:\n")
cat("   - Economic transmission mechanisms\n")
cat("   - Early warning indicators\n")
cat("   - Market-economic feedback loops\n")
cat("   - Systemic risk concentrations\n\n")

cat("APPLICATIONS:\n")
cat("- Economic forecasting and scenario analysis\n")
cat("- Investment strategy development\n")
cat("- Risk management and portfolio optimization\n")
cat("- Policy impact assessment\n")
cat("- Academic research in financial economics\n\n")

cat("=" %>% rep(80) %>% paste(collapse=""), "\n")
cat("Analysis completed at:", as.character(Sys.time()), "\n")
cat("=" %>% rep(80) %>% paste(collapse=""), "\n")

# Plot

In [ ]:
# Check data availability
cat("Checking data availability...\n")
cat("df_correlations exists:", exists("df_correlations"), "\n")
if (exists("df_correlations")) {
    cat("df_correlations rows:", nrow(df_correlations), "\n")
}

cat("strong_correlations exists:", exists("strong_correlations"), "\n")
if (exists("strong_correlations")) {
    cat("strong_correlations rows:", nrow(strong_correlations), "\n")
}

cat("nodes_df exists:", exists("nodes_df"), "\n")
if (exists("nodes_df")) {
    cat("nodes_df rows:", nrow(nodes_df), "\n")
}

cat("edges_df exists:", exists("edges_df"), "\n")
if (exists("edges_df")) {
    cat("edges_df rows:", nrow(edges_df), "\n")
}

In [ ]:
# Force create a simple test plot
if (exists("df_correlations") && nrow(df_correlations) > 0) {
    
    # Lower the threshold to ensure we get some data
    test_threshold <- 0.5
    
    test_network_data <- df_correlations %>%
        filter(abs(best_corr) >= test_threshold) %>%
        head(50)  # Limit for testing
    
    if (nrow(test_network_data) > 0) {
        cat("Creating test visualization with", nrow(test_network_data), "correlations\n")
        
        # Create simple NetworkD3 plot
        all_nodes <- unique(c(test_network_data$x1, test_network_data$x2))
        
        nodes_simple <- data.frame(
            name = all_nodes,
            group = 1
        )
        
        links_simple <- test_network_data %>%
            mutate(
                source = match(x1, all_nodes) - 1,  # 0-indexed
                target = match(x2, all_nodes) - 1,  # 0-indexed
                value = abs(best_corr) * 10
            ) %>%
            select(source, target, value)
        
        # Create the plot
        simple_network <- forceNetwork(
            Links = links_simple,
            Nodes = nodes_simple,
            Source = "source",
            Target = "target",
            Value = "value",
            NodeID = "name",
            Group = "group",
            opacity = 0.8,
            fontSize = 14,
            zoom = TRUE,
            legend = FALSE
        )
        
        print(simple_network)
        
    } else {
        cat("No data available even with threshold", test_threshold, "\n")
        cat("Max correlation:", max(abs(df_correlations$best_corr)), "\n")
    }
} else {
    cat("No correlation data available\n")
}

In [ ]:
# Debug correlation calculation
if (exists("df_filtered")) {
    cat("df_filtered rows:", nrow(df_filtered), "\n")
    cat("Unique series:", length(unique(df_filtered$series_id)), "\n")
    
    # Try a simple correlation test with just 2 series
    series_list <- unique(df_filtered$series_id)
    if (length(series_list) >= 2) {
        test_series1 <- series_list[1]
        test_series2 <- series_list[2]
        
        data1 <- df_filtered %>% filter(series_id == test_series1)
        data2 <- df_filtered %>% filter(series_id == test_series2)
        
        cat("Test series 1 (", test_series1, "):", nrow(data1), "observations\n")
        cat("Test series 2 (", test_series2, "):", nrow(data2), "observations\n")
        
        # Try simple correlation
        merged_test <- inner_join(
            data1 %>% select(date, value),
            data2 %>% select(date, value),
            by = "date",
            suffix = c(".x", ".y")
        )
        
        if (nrow(merged_test) > 10) {
            simple_cor <- cor(merged_test$value.x, merged_test$value.y, use = "complete.obs")
            cat("Simple correlation between", test_series1, "and", test_series2, ":", round(simple_cor, 3), "\n")
        }
    }
}

In [ ]:
# Enhanced network visualization - showing STRONG correlations only (|r| >= 0.7)
if (exists("df_correlations") && nrow(df_correlations) > 0) {
    
    cat("Creating network visualization with STRONG correlations only...\n")
    
    # Check what columns exist in df_correlations
    cat("Columns in df_correlations:", paste(colnames(df_correlations), collapse = ", "), "\n")
    cat("Total available correlations:", nrow(df_correlations), "\n")
    
    # Use threshold of 0.7 to show only strong correlations
    network_threshold <- 0.7  # Strong correlations only
    max_correlations <- 2000   # Increase limit to show more correlations
    
    # Show distribution of ALL correlations first
    correlation_distribution <- df_correlations %>%
        mutate(
            abs_corr = abs(best_corr),
            strength_category = case_when(
                abs_corr >= 0.8 ~ "Very Strong (≥0.8)",
                abs_corr >= 0.7 ~ "Strong (0.7-0.8)",
                abs_corr >= 0.6 ~ "Moderate-Strong (0.6-0.7)",
                abs_corr >= 0.4 ~ "Moderate (0.4-0.6)",
                abs_corr >= 0.2 ~ "Weak (0.2-0.4)",
                TRUE ~ "Very Weak (<0.2)"
            )
        ) %>%
        count(strength_category) %>%
        arrange(desc(n))
    
    cat("\nFULL correlation strength distribution:\n")
    print(correlation_distribution)
    
    # Filter to show only strong correlations (>= 0.7 or <= -0.7)
    network_data <- df_correlations %>%
        filter(abs(best_corr) >= network_threshold) %>%
        arrange(desc(abs(best_corr))) %>%
        head(max_correlations)  # Show top correlations above threshold
    
    cat("\nCorrelations with |r| >=", network_threshold, ":", 
        sum(abs(df_correlations$best_corr) >= network_threshold), "\n")
    
    if (nrow(network_data) > 0) {
        cat("Using", nrow(network_data), "STRONG correlations for visualization\n")
        
        # Show distribution of correlations being plotted
        plot_correlation_summary <- network_data %>%
            mutate(
                strength_category = case_when(
                    abs(best_corr) >= 0.9 ~ "Extremely Strong (≥0.9)",
                    abs(best_corr) >= 0.8 ~ "Very Strong (0.8-0.9)",
                    abs(best_corr) >= 0.7 ~ "Strong (0.7-0.8)",
                    TRUE ~ "Other"
                )
            ) %>%
            count(strength_category) %>%
            arrange(desc(n))
        
        cat("\nSTRONG correlations being plotted by strength:\n")
        print(plot_correlation_summary)
        
        # Method 1: igraph plot with STRONG correlations only
        library(igraph)
        
        # Create edges dataframe with proper column names
        edges_for_graph <- network_data %>%
            mutate(
                from = x1,
                to = x2,
                weight = abs(best_corr),
                original_corr = best_corr
            ) %>%
            select(from, to, weight, original_corr)
        
        # Create graph
        g <- graph_from_data_frame(edges_for_graph, directed = FALSE)
        
        # Set visual properties based on correlation strength - only strong correlations
        E(g)$color <- case_when(
            E(g)$original_corr >= 0.9 ~ "darkgreen",       # Extremely strong positive
            E(g)$original_corr >= 0.7 ~ "forestgreen",     # Strong positive
            E(g)$original_corr <= -0.9 ~ "darkred",        # Extremely strong negative
            E(g)$original_corr <= -0.7 ~ "red",            # Strong negative
            TRUE ~ "gray"                                   # Should not happen with our filter
        )
        
        E(g)$width <- scales::rescale(E(g)$weight, to = c(1, 6))  # Thicker edges for strong correlations
        E(g)$alpha <- scales::rescale(E(g)$weight, to = c(0.6, 1.0))  # High transparency for strong correlations
        
        # Node properties - size based on connections
        V(g)$size <- scales::rescale(degree(g), to = c(8, 25))  # Larger nodes for better visibility
        V(g)$color <- "lightblue"
        V(g)$label.cex <- 0.4  # Slightly larger labels since fewer nodes
        V(g)$label.color <- "black"
        
        # Create the plot with better layout
        par(mar = c(1, 1, 3, 1), bg = "white")
        
        # Use appropriate layout based on network size
        if (vcount(g) > 50) {
            layout_coords <- layout_with_lgl(g)  # Large Graph Layout
        } else {
            layout_coords <- layout_with_fr(g, niter = 2000)  # Fruchterman-Reingold with more iterations
        }
        
        plot(g,
             layout = layout_coords,
             main = paste("STRONG Economic Indicators Network\n", 
                         nrow(network_data), "correlations, |r| >=", network_threshold),
             vertex.label.dist = 0.3,
             edge.curved = 0.05,
             vertex.frame.color = "darkgray",
             vertex.frame.width = 0.5)
        
        # Enhanced legend for strong correlations only
        legend("topright", 
               legend = c("Extremely Strong Positive (≥0.9)", "Strong Positive (0.7-0.9)", 
                         "Extremely Strong Negative (≤-0.9)", "Strong Negative (-0.9 to -0.7)",
                         paste("Threshold: |r| ≥", network_threshold),
                         paste("Nodes:", vcount(g)),
                         paste("Edges:", ecount(g))),
               col = c("darkgreen", "forestgreen", "darkred", "red", "black", "black", "black"),
               lty = c(1, 1, 1, 1, 0, 0, 0), 
               lwd = c(3, 3, 3, 3, 0, 0, 0), 
               cex = 0.6, 
               bg = "white")
        
        # Method 2: Enhanced NetworkD3 with STRONG correlations only
        cat("\nCreating interactive NetworkD3 with STRONG correlations only...\n")
        
        # Prepare data for NetworkD3
        all_nodes <- unique(c(network_data$x1, network_data$x2))
        
        # Calculate node degrees for sizing
        node_degrees <- network_data %>%
            select(x1, x2) %>%
            pivot_longer(everything(), values_to = "node") %>%
            count(node, name = "degree")
        
        # Create node groups based on indicator type
        nodes_d3 <- data.frame(
            name = all_nodes,
            group = 1
        ) %>%
            left_join(node_degrees, by = c("name" = "node")) %>%
            mutate(
                degree = ifelse(is.na(degree), 1, degree),
                size = scales::rescale(degree, to = c(10, 40)),  # Larger range for strong correlations
                # Group nodes by type for coloring
                group = case_when(
                    name %in% unlist(fred_indicators$employment) ~ 1,
                    name %in% unlist(fred_indicators$inflation) ~ 2,
                    name %in% unlist(fred_indicators$rates) ~ 3,
                    name %in% unlist(fred_indicators$money) ~ 4,
                    name %in% unlist(fred_indicators$output) ~ 5,
                    name %in% unlist(fred_indicators$housing) ~ 6,
                    name %in% unlist(fred_indicators$sentiment) ~ 7,
                    name %in% unlist(fred_indicators$risk) ~ 8,
                    name %in% unlist(stock_symbols$indices) ~ 9,
                    name %in% unlist(stock_symbols) ~ 10,
                    TRUE ~ 11
                )
            )
        
        links_d3 <- network_data %>%
            mutate(
                source = match(x1, all_nodes) - 1,  # 0-indexed
                target = match(x2, all_nodes) - 1,  # 0-indexed
                value = abs(best_corr) * 20,  # Scale for visibility - stronger scaling for strong correlations
                correlation = round(best_corr, 3),
                lag = best_lag
            ) %>%
            select(source, target, value, correlation, lag)
        
        # Create interactive network with STRONG correlations only
        interactive_network <- forceNetwork(
            Links = links_d3,
            Nodes = nodes_d3,
            Source = "source",
            Target = "target",
            Value = "value",
            NodeID = "name",
            Group = "group",
            opacity = 0.8,  # Higher opacity for strong correlations
            fontSize = 12,   # Larger font for better readability
            fontFamily = "Arial",
            linkDistance = 80,  # More space between nodes for clarity
            linkWidth = JS("function(d){return Math.sqrt(d.value/2);}"),
            charge = -200,  # Moderate repulsion for clear layout
            zoom = TRUE,
            legend = TRUE,
            width = 1400,
            height = 900,
            colourScale = JS('d3.scaleOrdinal()
                .domain([1,2,3,4,5,6,7,8,9,10,11])
                .range(["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf","#aec7e8"])')
        )
        
        print(interactive_network)
        
        # Method 3: Summary statistics for STRONG correlations network
        cat("\nSTRONG CORRELATIONS Network Statistics:\n")
        cat("- Total STRONG correlations plotted:", nrow(network_data), "\n")
        cat("- Correlation threshold used:", network_threshold, "\n")
        cat("- Strong positive correlations (≥0.7):", sum(network_data$best_corr >= 0.7), "\n")
        cat("- Strong negative correlations (≤-0.7):", sum(network_data$best_corr <= -0.7), "\n")
        cat("- Correlation range:", round(min(network_data$best_corr), 3), "to", round(max(network_data$best_corr), 3), "\n")
        cat("- Number of unique indicators:", length(all_nodes), "\n")
        cat("- Network density:", round(ecount(g) / (vcount(g) * (vcount(g) - 1) / 2), 4), "\n")
        cat("- Average degree:", round(mean(degree(g)), 2), "\n")
        cat("- Maximum degree:", max(degree(g)), "\n")
        
        # Top hub nodes (most connected) from STRONG correlations only
        top_hubs_strong <- node_degrees %>%
            arrange(desc(degree)) %>%
            head(15)
        
        cat("\nTop 15 most connected indicators (STRONG correlations only):\n")
        print(top_hubs_strong)
        
        # Show correlation strength breakdown for strong correlations
        cat("\nSTRONG correlation strength breakdown:\n")
        strength_breakdown_strong <- network_data %>%
            summarise(
                extremely_strong_pos = sum(best_corr >= 0.9),
                very_strong_pos = sum(best_corr >= 0.8 & best_corr < 0.9),
                strong_pos = sum(best_corr >= 0.7 & best_corr < 0.8),
                strong_neg = sum(best_corr <= -0.7 & best_corr > -0.8),
                very_strong_neg = sum(best_corr <= -0.8 & best_corr > -0.9),
                extremely_strong_neg = sum(best_corr <= -0.9)
            )
        
        print(strength_breakdown_strong)
        
        # Additional analysis: Show most extreme correlations
        extreme_correlations <- network_data %>%
            arrange(desc(abs(best_corr))) %>%
            head(10) %>%
            select(x1, x2, best_corr, best_lag, n_obs) %>%
            mutate(best_corr = round(best_corr, 4))
        
        cat("\nTop 10 strongest correlations:\n")
        print(extreme_correlations)
        
    } else {
        cat("No STRONG correlations found with threshold", network_threshold, "\n")
        if (nrow(df_correlations) > 0) {
            strong_count <- sum(abs(df_correlations$best_corr) >= 0.7)
            cat("Available strong correlations (|r| >= 0.7):", strong_count, "\n")
            cat("Max correlation:", round(max(abs(df_correlations$best_corr)), 3), "\n")
        }
    }
    
} else {
    cat("No correlation data available for visualization\n")
}

In [ ]:
# Enhanced Network Visualization Test
cat("Testing network visualization components...\n")

# Test 1: Basic data availability
if (exists("df_correlations") && nrow(df_correlations) > 0) {
    cat("✓ Correlation data available:", nrow(df_correlations), "correlations\n")
} else {
    cat("✗ No correlation data available\n")
}

# Test 2: Strong correlations
if (exists("strong_correlations") && nrow(strong_correlations) > 0) {
    cat("✓ Strong correlations available:", nrow(strong_correlations), "strong correlations\n")
} else {
    cat("✗ No strong correlations available\n")
}

# Test 3: Network components
if (exists("nodes_df") && exists("edges_df")) {
    cat("✓ Network components created - Nodes:", nrow(nodes_df), "Edges:", nrow(edges_df), "\n")
    
    # Show sample of nodes and edges
    cat("\nSample nodes:\n")
    print(head(nodes_df %>% select(id, label, node_type, degree), 5))
    
    cat("\nSample edges:\n")
    print(head(edges_df %>% select(from, to, weight), 5))
    
    # Validate network structure
    all_edge_nodes <- unique(c(edges_df$from, edges_df$to))
    missing_nodes <- setdiff(all_edge_nodes, nodes_df$id)
    
    if (length(missing_nodes) == 0) {
        cat("✓ Network structure is valid - all edge endpoints have corresponding nodes\n")
    } else {
        cat("✗ Missing nodes in network:", length(missing_nodes), "nodes\n")
        cat("Missing:", paste(head(missing_nodes, 5), collapse = ", "), "\n")
    }
    
    # Node type distribution
    node_type_dist <- table(nodes_df$node_type)
    cat("\nNode type distribution:\n")
    print(node_type_dist)
    
    # Edge strength distribution
    cat("\nEdge strength distribution:\n")
    print(summary(edges_df$weight))
    
} else {
    cat("✗ Network components not created\n")
}

# Test 4: Create a minimal working network for testing
if (exists("df_correlations") && nrow(df_correlations) > 0) {
    
    cat("\nCreating minimal test network...\n")
    
    # Get top 10 most correlated pairs
    test_correlations <- df_correlations %>%
        arrange(desc(abs(best_corr))) %>%
        head(20) %>%
        filter(abs(best_corr) >= 0.8)
    
    if (nrow(test_correlations) > 0) {
        # Create test nodes
        test_node_ids <- unique(c(test_correlations$x1, test_correlations$x2))
        test_nodes <- data.frame(
            id = test_node_ids,
            label = test_node_ids,
            title = paste("Test node:", test_node_ids),
            color = "#69b3a2",
            size = 20
        )
        
        # Create test edges
        test_edges <- test_correlations %>%
            mutate(
                from = x1,
                to = x2,
                weight = abs(best_corr),
                color = ifelse(best_corr > 0, "green", "red"),
                width = scales::rescale(abs(best_corr), to = c(1, 5)),
                title = paste("Correlation:", round(best_corr, 3))
            ) %>%
            select(from, to, weight, color, width, title)
        
        cat("Test network: ", nrow(test_nodes), "nodes,", nrow(test_edges), "edges\n")
        
        # Create test visualization
        test_network <- visNetwork(test_nodes, test_edges, height = "400px") %>%
            visOptions(highlightNearest = TRUE) %>%
            visInteraction(navigationButtons = TRUE) %>%
            visLayout(randomSeed = 123)
        
        cat("\nTest network visualization:\n")
        print(test_network)
        
    } else {
        cat("No suitable correlations found for test network\n")
    }
}

# Test 5: Alternative visualization using networkD3
if (require(networkD3, quietly = TRUE) && exists("df_correlations") && nrow(df_correlations) > 0) {
    
    cat("\nTesting NetworkD3 visualization...\n")
    
    # Prepare data for networkD3
    strong_corr_threshold <- 0.8
    network_data_d3 <- df_correlations %>%
        filter(abs(best_corr) >= strong_corr_threshold) %>%
        head(50)  # Limit for testing
    
    if (nrow(network_data_d3) > 0) {
        # Create nodes for networkD3 (0-indexed)
        all_nodes_d3 <- unique(c(network_data_d3$x1, network_data_d3$x2))
        nodes_d3 <- data.frame(
            name = all_nodes_d3,
            group = 1,
            size = 10
        )
        
        # Create links for networkD3
        links_d3 <- network_data_d3 %>%
            left_join(data.frame(name = all_nodes_d3, source_id = 0:(length(all_nodes_d3)-1)), 
                     by = c("x1" = "name")) %>%
            left_join(data.frame(name = all_nodes_d3, target_id = 0:(length(all_nodes_d3)-1)), 
                     by = c("x2" = "name")) %>%
            mutate(value = abs(best_corr) * 10) %>%
            select(source = source_id, target = target_id, value)
        
        # Create NetworkD3 visualization
        networkD3_plot <- forceNetwork(
            Links = links_d3,
            Nodes = nodes_d3,
            Source = "source",
            Target = "target",
            Value = "value",
            NodeID = "name",
            Group = "group",
            opacity = 0.8,
            zoom = TRUE,
            fontSize = 12
        )
        
        cat("NetworkD3 test visualization:\n")
        print(networkD3_plot)
    }
}

cat("\nVisualization test complete!\n")

## 13. Beta Coefficient Analysis

Beta coefficients measure the systematic risk of stocks relative to economic indicators. This section calculates beta values between stock prices and economic indicators to understand how sensitive each stock is to changes in economic conditions.

In [ ]:
# Function to calculate beta coefficient between a stock and an economic indicator
calculate_beta_coefficient <- function(stock_data, indicator_data, min_obs = 200) {
    tryCatch({
        # Merge data on common dates
        merged <- inner_join(
            stock_data %>% select(date, value) %>% rename(stock_price = value),
            indicator_data %>% select(date, value) %>% rename(indicator_value = value),
            by = "date"
        ) %>%
        arrange(date) %>%
        filter(
            !is.na(stock_price), !is.na(indicator_value),
            is.finite(stock_price), is.finite(indicator_value)
        )
        
        if (nrow(merged) < min_obs) {
            return(NULL)  # Not enough data
        }
        
        # Calculate returns/changes
        merged <- merged %>%
            mutate(
                stock_return = (stock_price - lag(stock_price)) / lag(stock_price),
                indicator_change = (indicator_value - lag(indicator_value)) / lag(indicator_value)
            ) %>%
            filter(!is.na(stock_return), !is.na(indicator_change),
                   is.finite(stock_return), is.finite(indicator_change))
        
        if (nrow(merged) < min_obs) {
            return(NULL)
        }
        
        # Calculate beta using linear regression: stock_return = alpha + beta * indicator_change
        if (var(merged$indicator_change, na.rm = TRUE) > 0) {
            beta_model <- lm(stock_return ~ indicator_change, data = merged)
            beta_coeff <- as.numeric(coef(beta_model)[2])  # Beta coefficient
            alpha_coeff <- as.numeric(coef(beta_model)[1])  # Alpha (intercept)
            r_squared <- summary(beta_model)$r.squared
            p_value <- summary(beta_model)$coefficients[2, 4]  # P-value for beta
            
            # Also calculate correlation for comparison
            correlation <- cor(merged$stock_return, merged$indicator_change, use = "complete.obs")
            
            return(data.frame(
                beta = beta_coeff,
                alpha = alpha_coeff,
                correlation = correlation,
                r_squared = r_squared,
                p_value = p_value,
                n_obs = nrow(merged),
                stringsAsFactors = FALSE
            ))
        } else {
            return(NULL)
        }
        
    }, error = function(e) {
        return(NULL)
    })
}

# Calculate beta coefficients for all stock-indicator pairs
if (exists("df_filtered") && nrow(df_filtered) > 0) {
    
    cat("Calculating beta coefficients between stocks and economic indicators...\n")
    
    # Separate stocks and economic indicators
    stock_series <- df_filtered %>%
        filter(series_id %in% unlist(stock_symbols)) %>%
        pull(series_id) %>%
        unique()
    
    fred_series <- df_filtered %>%
        filter(series_id %in% unlist(fred_indicators)) %>%
        pull(series_id) %>%
        unique()
    
    cat("Stock series:", length(stock_series), "\n")
    cat("FRED indicators:", length(fred_series), "\n")
    
    # Create all combinations of stocks and indicators
    stock_indicator_combinations <- expand.grid(
        stock = stock_series,
        indicator = fred_series,
        stringsAsFactors = FALSE
    )
    
    cat("Total stock-indicator combinations:", nrow(stock_indicator_combinations), "\n")
    
    # Calculate beta coefficients in batches
    batch_size <- 500  # Smaller batch size for beta calculations
    n_batches <- ceiling(nrow(stock_indicator_combinations) / batch_size)
    
    beta_results <- list()
    
    for (i in 1:n_batches) {
        start_idx <- (i - 1) * batch_size + 1
        end_idx <- min(i * batch_size, nrow(stock_indicator_combinations))
        
        cat("Processing beta batch", i, "of", n_batches, "\n")
        
        batch_combinations <- stock_indicator_combinations[start_idx:end_idx, ]
        
        batch_betas <- mapply(
            function(stock_id, indicator_id) {
                stock_data <- df_filtered %>% filter(series_id == stock_id)
                indicator_data <- df_filtered %>% filter(series_id == indicator_id)
                
                result <- calculate_beta_coefficient(stock_data, indicator_data)
                if (!is.null(result)) {
                    result$stock_symbol <- stock_id
                    result$economic_indicator <- indicator_id
                    return(result)
                } else {
                    return(NULL)
                }
            },
            batch_combinations$stock,
            batch_combinations$indicator,
            SIMPLIFY = FALSE
        )
        
        # Remove NULL results
        batch_betas <- batch_betas[!sapply(batch_betas, is.null)]
        
        if (length(batch_betas) > 0) {
            beta_results[[i]] <- rbindlist(batch_betas)
        }
        
        # Add small delay to prevent overloading
        if (i %% 5 == 0) {
            Sys.sleep(0.5)
        }
    }
    
    # Combine all beta results
    if (length(beta_results) > 0) {
        df_beta_coefficients <- rbindlist(beta_results)
        
        # Add additional categorization
        df_beta_coefficients <- df_beta_coefficients %>%
            mutate(
                # Stock categorization
                stock_type = case_when(
                    stock_symbol %in% unlist(stock_symbols$indices) ~ "Market Index",
                    stock_symbol %in% unlist(stock_symbols$tech) ~ "Tech Stock",
                    stock_symbol %in% unlist(stock_symbols$financial) ~ "Financial Stock",
                    stock_symbol %in% unlist(stock_symbols$healthcare) ~ "Healthcare Stock",
                    stock_symbol %in% unlist(stock_symbols$consumer_disc) ~ "Consumer Discretionary",
                    stock_symbol %in% unlist(stock_symbols$consumer_staples) ~ "Consumer Staples",
                    stock_symbol %in% unlist(stock_symbols$industrial) ~ "Industrial Stock",
                    stock_symbol %in% unlist(stock_symbols$energy) ~ "Energy Stock",
                    stock_symbol %in% unlist(stock_symbols$utilities) ~ "Utilities Stock",
                    stock_symbol %in% unlist(stock_symbols$real_estate) ~ "Real Estate Stock",
                    TRUE ~ "Other Stock"
                ),
                # Economic indicator categorization
                indicator_category = case_when(
                    economic_indicator %in% unlist(fred_indicators$employment) ~ "Employment",
                    economic_indicator %in% unlist(fred_indicators$inflation) ~ "Inflation",
                    economic_indicator %in% unlist(fred_indicators$rates) ~ "Interest Rates",
                    economic_indicator %in% unlist(fred_indicators$money) ~ "Money Supply",
                    economic_indicator %in% unlist(fred_indicators$output) ~ "Economic Output",
                    economic_indicator %in% unlist(fred_indicators$housing) ~ "Housing",
                    economic_indicator %in% unlist(fred_indicators$sentiment) ~ "Sentiment",
                    economic_indicator %in% unlist(fred_indicators$risk) ~ "Financial Risk",
                    economic_indicator %in% unlist(fred_indicators$international) ~ "International",
                    TRUE ~ "Other"
                ),
                # Beta interpretation
                beta_interpretation = case_when(
                    abs(beta) < 0.5 ~ "Low Sensitivity",
                    abs(beta) < 1.0 ~ "Moderate Sensitivity", 
                    abs(beta) < 1.5 ~ "High Sensitivity",
                    TRUE ~ "Very High Sensitivity"
                ),
                # Statistical significance
                significant = p_value < 0.05
            ) %>%
            # Filter for statistically significant results
            filter(significant == TRUE, n_obs >= 200) %>%
            arrange(desc(abs(beta)))
        
        cat("\nBeta coefficient analysis complete!\n")
        cat("Total significant beta coefficients calculated:", nrow(df_beta_coefficients), "\n")
        cat("Average |beta|:", round(mean(abs(df_beta_coefficients$beta), na.rm = TRUE), 3), "\n")
        
        # Show top beta coefficients
        top_betas <- df_beta_coefficients %>%
            head(20) %>%
            select(stock_symbol, stock_type, economic_indicator, indicator_category, 
                   beta, correlation, r_squared, p_value, n_obs) %>%
            mutate(
                beta = round(beta, 4),
                correlation = round(correlation, 3),
                r_squared = round(r_squared, 3),
                p_value = round(p_value, 6)
            )
        
        DT::datatable(top_betas, 
                      caption = "Top 20 Beta Coefficients (Stock Sensitivity to Economic Indicators)",
                      options = list(pageLength = 20, scrollX = TRUE))
        
    } else {
        cat("No significant beta coefficients calculated\n")
    }
    
} else {
    cat("No filtered data available for beta analysis\n")
}

## 14. Comprehensive Correlation and Beta Coefficients Table

This section creates comprehensive tables showing both correlations and beta coefficients for easy comparison and analysis.

In [ ]:
# Create comprehensive correlation and beta coefficients table
if (exists("df_beta_coefficients") && exists("stock_relationships")) {
    
    cat("Creating comprehensive correlation and beta coefficients table...\n")
    
    # Combine correlation data with beta coefficients
    comprehensive_table <- df_beta_coefficients %>%
        left_join(
            stock_relationships %>%
                select(stock_symbol, economic_indicator, best_corr, best_lag, 
                       zero_lag_corr, correlation_type, lag_type),
            by = c("stock_symbol", "economic_indicator")
        ) %>%
        # Add additional metrics and clean up
        mutate(
            # Round all numeric values for better display
            beta = round(beta, 4),
            alpha = round(alpha, 6),
            correlation_returns = round(correlation, 3),  # Correlation of returns
            correlation_levels = round(ifelse(is.na(best_corr), zero_lag_corr, best_corr), 3),  # Correlation of levels
            r_squared = round(r_squared, 3),
            p_value = round(p_value, 6),
            best_lag = ifelse(is.na(best_lag), 0, best_lag),
            
            # Create interpretive fields
            beta_magnitude = case_when(
                abs(beta) < 0.25 ~ "Very Low",
                abs(beta) < 0.5 ~ "Low",
                abs(beta) < 1.0 ~ "Moderate",
                abs(beta) < 1.5 ~ "High",
                abs(beta) < 2.0 ~ "Very High",
                TRUE ~ "Extreme"
            ),
            
            correlation_strength = case_when(
                abs(correlation_levels) < 0.3 ~ "Weak",
                abs(correlation_levels) < 0.5 ~ "Moderate", 
                abs(correlation_levels) < 0.7 ~ "Strong",
                abs(correlation_levels) < 0.9 ~ "Very Strong",
                TRUE ~ "Extremely Strong"
            ),
            
            economic_sensitivity = case_when(
                r_squared < 0.1 ~ "Low Explained",
                r_squared < 0.25 ~ "Moderate Explained",
                r_squared < 0.5 ~ "High Explained",
                TRUE ~ "Very High Explained"
            )
        ) %>%
        # Select and rename columns for final display
        select(
            `Stock Symbol` = stock_symbol,
            `Stock Type` = stock_type,
            `Economic Indicator` = economic_indicator,
            `Indicator Category` = indicator_category,
            `Beta Coefficient` = beta,
            `Beta Magnitude` = beta_magnitude,
            `Alpha (Intercept)` = alpha,
            `Correlation (Returns)` = correlation_returns,
            `Correlation (Levels)` = correlation_levels,
            `Correlation Strength` = correlation_strength,
            `R-Squared` = r_squared,
            `Economic Sensitivity` = economic_sensitivity,
            `P-Value` = p_value,
            `Lag (Days)` = best_lag,
            `Lag Type` = lag_type,
            `Observations` = n_obs,
            `Beta Interpretation` = beta_interpretation
        ) %>%
        arrange(desc(abs(`Beta Coefficient`)))
    
    cat("Comprehensive table created with", nrow(comprehensive_table), "entries\n")
    
    # Display the comprehensive table
    DT::datatable(comprehensive_table, 
                  caption = "Comprehensive Stock-Economic Indicator Analysis: Correlations & Beta Coefficients",
                  options = list(
                      pageLength = 25, 
                      scrollX = TRUE,
                      columnDefs = list(
                          list(className = 'dt-center', targets = c(4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15))
                      )
                  )) %>%
        formatStyle(
            "Beta Coefficient",
            background = styleColorBar(range(comprehensive_table$`Beta Coefficient`), "lightcoral"),
            backgroundSize = "100% 90%",
            backgroundRepeat = "no-repeat",
            backgroundPosition = "center"
        ) %>%
        formatStyle(
            "R-Squared",
            background = styleColorBar(c(0, 1), "lightblue"),
            backgroundSize = "100% 90%",
            backgroundRepeat = "no-repeat", 
            backgroundPosition = "center"
        ) %>%
        formatStyle(
            "Correlation (Levels)",
            background = styleColorBar(c(-1, 1), "lightgreen"),
            backgroundSize = "100% 90%",
            backgroundRepeat = "no-repeat",
            backgroundPosition = "center"
        )
    
} else {
    cat("Beta coefficients or stock relationships not available for comprehensive table\n")
}

In [ ]:
# Summary statistics and analysis of beta coefficients
if (exists("df_beta_coefficients")) {
    
    cat("\n" , "=" %>% rep(80) %>% paste(collapse=""), "\n")
    cat("BETA COEFFICIENT ANALYSIS SUMMARY\n") 
    cat("=" %>% rep(80) %>% paste(collapse=""), "\n\n")
    
    # Overall beta statistics
    cat("OVERALL BETA STATISTICS:\n")
    cat("Total significant beta coefficients:", nrow(df_beta_coefficients), "\n")
    cat("Beta coefficient range:", round(min(df_beta_coefficients$beta), 4), "to", round(max(df_beta_coefficients$beta), 4), "\n")
    cat("Average |beta|:", round(mean(abs(df_beta_coefficients$beta)), 4), "\n")
    cat("Median |beta|:", round(median(abs(df_beta_coefficients$beta)), 4), "\n")
    cat("Standard deviation of beta:", round(sd(df_beta_coefficients$beta), 4), "\n\n")
    
    # Beta distribution by magnitude
    beta_distribution <- table(df_beta_coefficients$beta_interpretation)
    cat("BETA MAGNITUDE DISTRIBUTION:\n")
    print(beta_distribution)
    cat("\n")
    
    # Beta by stock type
    beta_by_stock_type <- df_beta_coefficients %>%
        group_by(stock_type) %>%
        summarise(
            count = n(),
            avg_abs_beta = round(mean(abs(beta)), 4),
            median_abs_beta = round(median(abs(beta)), 4),
            max_abs_beta = round(max(abs(beta)), 4),
            avg_r_squared = round(mean(r_squared), 3),
            .groups = 'drop'
        ) %>%
        arrange(desc(avg_abs_beta))
    
    cat("BETA SENSITIVITY BY STOCK TYPE:\n")
    print(beta_by_stock_type)
    cat("\n")
    
    # Beta by economic indicator category
    beta_by_indicator <- df_beta_coefficients %>%
        group_by(indicator_category) %>%
        summarise(
            count = n(),
            avg_abs_beta = round(mean(abs(beta)), 4),
            median_abs_beta = round(median(abs(beta)), 4),
            max_abs_beta = round(max(abs(beta)), 4),
            avg_r_squared = round(mean(r_squared), 3),
            .groups = 'drop'
        ) %>%
        arrange(desc(avg_abs_beta))
    
    cat("BETA SENSITIVITY BY ECONOMIC INDICATOR CATEGORY:\n")
    print(beta_by_indicator)
    cat("\n")
    
    # Top 10 highest beta coefficients (positive and negative)
    top_positive_betas <- df_beta_coefficients %>%
        arrange(desc(beta)) %>%
        head(5) %>%
        select(stock_symbol, economic_indicator, beta, correlation, r_squared) %>%
        mutate(
            beta = round(beta, 4),
            correlation = round(correlation, 3),
            r_squared = round(r_squared, 3)
        )
    
    top_negative_betas <- df_beta_coefficients %>%
        arrange(beta) %>%
        head(5) %>%
        select(stock_symbol, economic_indicator, beta, correlation, r_squared) %>%
        mutate(
            beta = round(beta, 4),
            correlation = round(correlation, 3),
            r_squared = round(r_squared, 3)
        )
    
    cat("TOP 5 HIGHEST POSITIVE BETA COEFFICIENTS:\n")
    print(top_positive_betas)
    cat("\n")
    
    cat("TOP 5 HIGHEST NEGATIVE BETA COEFFICIENTS:\n")
    print(top_negative_betas)
    cat("\n")
    
    # Correlation vs Beta analysis
    if (exists("comprehensive_table")) {
        
        # Create scatter plot of correlation vs beta
        if (require(ggplot2, quietly = TRUE)) {
            
            correlation_vs_beta <- comprehensive_table %>%
                filter(!is.na(`Correlation (Levels)`), !is.na(`Beta Coefficient`))
            
            if (nrow(correlation_vs_beta) > 0) {
            
                corr_beta_plot <- ggplot(correlation_vs_beta, 
                                       aes(x = `Correlation (Levels)`, y = `Beta Coefficient`, 
                                           color = `Stock Type`, size = `R-Squared`)) +
                    geom_point(alpha = 0.7) +
                    geom_smooth(method = "lm", se = FALSE, color = "black", linetype = "dashed") +
                    scale_size_continuous(range = c(1, 5), name = "R²") +
                    scale_color_brewer(type = "qual", palette = "Set3", name = "Stock Type") +
                    labs(
                        title = "Correlation vs Beta Coefficient Analysis",
                        subtitle = "Relationship between level correlations and return sensitivities",
                        x = "Correlation (Price Levels)",
                        y = "Beta Coefficient (Return Sensitivity)",
                        caption = "Each point represents a stock-economic indicator pair"
                    ) +
                    theme_minimal() +
                    theme(
                        legend.position = "right",
                        legend.box = "vertical"
                    ) +
                    geom_hline(yintercept = 0, linetype = "solid", alpha = 0.3) +
                    geom_vline(xintercept = 0, linetype = "solid", alpha = 0.3)
                
                print(corr_beta_plot)
                
                # Calculate correlation between correlation and beta
                corr_vs_beta_corr <- cor(correlation_vs_beta$`Correlation (Levels)`, 
                                        correlation_vs_beta$`Beta Coefficient`, 
                                        use = "complete.obs")
                
                cat("CORRELATION BETWEEN CORRELATION AND BETA:", round(corr_vs_beta_corr, 4), "\n\n")
            }
        }
    }
    
    # Key insights
    cat("KEY INSIGHTS FROM BETA ANALYSIS:\n")
    cat("1. Beta coefficients measure stock return sensitivity to economic indicator changes\n")
    cat("2. Higher |beta| indicates greater economic sensitivity and systematic risk\n")
    cat("3. Positive beta suggests stock moves in same direction as economic indicator\n")
    cat("4. Negative beta suggests inverse relationship with economic indicator\n")
    cat("5. R-squared shows percentage of stock return variance explained by indicator\n\n")
    
    cat("=" %>% rep(80) %>% paste(collapse=""), "\n")
    
} else {
    cat("Beta coefficients not available for summary analysis\n")
}

In [ ]:
# Export enhanced data including beta coefficients
if (exists("df_beta_coefficients") && exists("comprehensive_table")) {
    
    cat("Exporting enhanced analysis with beta coefficients...\n")
    
    # Update the export data to include beta coefficients
    enhanced_export_data <- list(
        metadata = list(
            created_date = Sys.Date(),
            analysis_period = paste(start_date, "to", end_date),
            total_indicators = length(unique(df_filtered$series_id)),
            correlation_threshold = significance_threshold,
            fred_indicators = fred_indicators,
            stock_symbols = stock_symbols,
            beta_analysis_included = TRUE
        ),
        nodes = if(exists("nodes_df")) nodes_df else NULL,
        edges = if(exists("edges_df")) edges_df else NULL,
        correlations = if(exists("strong_correlations")) strong_correlations else NULL,
        leadership = if(exists("leadership_analysis")) leadership_analysis else NULL,
        stock_relationships = if(exists("stock_relationships")) stock_relationships else NULL,
        beta_coefficients = df_beta_coefficients,
        comprehensive_analysis = comprehensive_table
    )
    
    # Save as RDS for R users
    saveRDS(enhanced_export_data, "knowledgegraph/enhanced_knowledge_graph_data.rds")
    
    # Save beta coefficients as CSV
    write.csv(df_beta_coefficients, "knowledgegraph/beta_coefficients.csv", row.names = FALSE)
    
    # Save comprehensive table as CSV
    write.csv(comprehensive_table, "knowledgegraph/comprehensive_correlation_beta_analysis.csv", row.names = FALSE)
    
    # Create summary statistics file
    beta_summary <- data.frame(
        Metric = c(
            "Total Beta Coefficients",
            "Average |Beta|",
            "Median |Beta|", 
            "Max |Beta|",
            "Min Beta",
            "Max Beta",
            "Average R-Squared",
            "Significant Relationships (%)"
        ),
        Value = c(
            nrow(df_beta_coefficients),
            round(mean(abs(df_beta_coefficients$beta)), 4),
            round(median(abs(df_beta_coefficients$beta)), 4),
            round(max(abs(df_beta_coefficients$beta)), 4),
            round(min(df_beta_coefficients$beta), 4),
            round(max(df_beta_coefficients$beta), 4),
            round(mean(df_beta_coefficients$r_squared), 4),
            round(mean(df_beta_coefficients$significant) * 100, 1)
        )
    )
    
    write.csv(beta_summary, "knowledgegraph/beta_analysis_summary.csv", row.names = FALSE)
    
    cat("\nEnhanced Knowledge Graph Export Complete!\n")
    cat("New files saved to knowledgegraph/ directory:\n")
    cat("- enhanced_knowledge_graph_data.rds (complete enhanced R data)\n")
    cat("- beta_coefficients.csv (all beta coefficients)\n") 
    cat("- comprehensive_correlation_beta_analysis.csv (combined analysis)\n")
    cat("- beta_analysis_summary.csv (summary statistics)\n")
    cat("\nPrevious files also updated with beta analysis integration.\n")
    
} else {
    cat("Beta coefficients or comprehensive table not available for export\n")
}